# Full RAG PDF Chatbot — FAISS + MiniLM + Mistral + Gradio

This notebook provides a complete, ready-to-run Retrieval-Augmented Generation (RAG) pipeline with:

- PDF upload (safe handling)
- Text chunking and MiniLM embeddings (`all-MiniLM-L6-v2`)
- FAISS vector store (CPU)
- Mistral-7B text generation via `transformers` pipeline (uses GPU if available)
- Conversational memory using LangChain `ConversationBufferMemory`
- Gradio chat UI (role/message format)

In [1]:
# Install required packages. Run this cell once.
# If you already have these installed, you can skip or run to ensure latest compatible versions.
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118 || true
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu gradio pypdf langchain langchain-huggingface langchain-community


ERROR: To modify pip, please run the following command:
C:\Users\ADMIN\anaconda3\python.exe -m pip install -q --upgrade pip


In [1]:
# Imports and persistent paths
import os, time, shutil, pickle
from pathlib import Path
import json
import traceback

import gradio as gr
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.schema import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS as LCFAISS
from langchain.llms import HuggingFacePipeline

# Persistent storage (change these paths if you prefer)
BASE_DIR = Path(r"C:\Users\ADMIN\Documents\rag_chatbot") if os.name == "nt" else Path.cwd() / "rag_chatbot"
BASE_DIR.mkdir(parents=True, exist_ok=True)
DOCS_PATH = BASE_DIR / "documents.pkl"
INDEX_PATH = BASE_DIR / "faiss_index.bin"
UPLOAD_DIR = BASE_DIR / "uploaded_pdfs"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Globals (will be initialized below)
embedder = None
faiss_index = None
documents = []
retriever = None
qa_chain = None
llm = None
tokenizer = None
gen_pipeline = None

print('Paths:') 
print(' BASE_DIR =', BASE_DIR)
print(' DOCS_PATH =', DOCS_PATH)
print(' INDEX_PATH =', INDEX_PATH)
print(' UPLOAD_DIR =', UPLOAD_DIR)


Paths:
 BASE_DIR = C:\Users\ADMIN\Documents\rag_chatbot
 DOCS_PATH = C:\Users\ADMIN\Documents\rag_chatbot\documents.pkl
 INDEX_PATH = C:\Users\ADMIN\Documents\rag_chatbot\faiss_index.bin
 UPLOAD_DIR = C:\Users\ADMIN\Documents\rag_chatbot\uploaded_pdfs


In [2]:
# Load embedding model (MiniLM) and initialize FAISS index if present
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embedding_dim = 384  # miniLM output dimension

# Load persisted documents & index if available
if INDEX_PATH.exists():
    try:
        faiss_index = faiss.read_index(str(INDEX_PATH))
        print("Loaded FAISS index from", INDEX_PATH)
    except Exception as e:
        print("Could not read FAISS index:", e)
        faiss_index = faiss.IndexFlatL2(embedding_dim)
else:
    faiss_index = faiss.IndexFlatL2(embedding_dim)

if DOCS_PATH.exists():
    try:
        with open(DOCS_PATH, "rb") as f:
            documents = pickle.load(f)
        print(f"Loaded {len(documents)} document chunks from {DOCS_PATH}")
    except Exception as e:
        print("Could not load documents.pkl:", e)
        documents = []
else:
    documents = []


Loaded FAISS index from C:\Users\ADMIN\Documents\rag_chatbot\faiss_index.bin
Loaded 4059 document chunks from C:\Users\ADMIN\Documents\rag_chatbot\documents.pkl


In [3]:
# Safe PDF handling: save uploaded temp file to disk, extract text, chunk, embed, update FAISS
def process_pdf(uploaded_file):
    global faiss_index, documents, retriever, qa_chain
    try:
        if uploaded_file is None:
            return " No file received."
        
        # save uploaded file to uploads dir (ensures upload stream finished)
        ts = int(time.time() * 1000)
        filename = os.path.basename(uploaded_file.name)
        safe_name = f"{ts}_{filename}"
        safe_path = UPLOAD_DIR / safe_name
        with open(uploaded_file.name, "rb") as src, open(safe_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
        
        # extract text using pypdf (page-by-page)
        reader = PdfReader(str(safe_path))
        pages = []
        for p in reader.pages:
            txt = p.extract_text()
            if txt:
                pages.append(txt)
        full_text = "\n\n".join(pages).strip()
        if not full_text:
            return "No text could be extracted from this PDF."
        
        # chunk into ~500-char pieces (simple approach)
        chunk_size = 500
        chunks = [full_text[i:i+chunk_size] for i in range(0, len(full_text), chunk_size)]
        
        # embed chunks (batch to avoid memory spike)
        batch_size = 64
        all_emb = []
        for i in range(0, len(chunks), batch_size):
            batch = chunks[i:i+batch_size]
            emb = embedder.encode(batch, convert_to_numpy=True)
            all_emb.append(emb)
        all_emb = np.vstack(all_emb).astype('float32')
        
        # add into FAISS
        faiss_index.add(all_emb)
        
        # persist docs and index
        documents.extend(chunks)
        with open(DOCS_PATH, "wb") as f:
            pickle.dump(documents, f)
        faiss.write_index(faiss_index, str(INDEX_PATH))
        
        # rebuild retriever & QA chain (only if LLM is available)
        retriever = build_retriever()
        if llm is not None:  # Only build QA chain if LLM is loaded
            qa_chain = build_qa_chain()
            return f"Uploaded {len(chunks)} chunks from {filename} and built QA chain."
        else:
            return f"Uploaded {len(chunks)} chunks from {filename}. Load model to enable chatting."
        
    except Exception as e:
        traceback.print_exc()
        return f" Error processing PDF: {e}"

In [4]:
# Load text-generation model (Mistral) using transformers pipeline.
# This cell tries to use 8-bit quantization if available to reduce VRAM usage.
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

def load_model(model_name):
    global tokenizer, gen_pipeline, llm
    try:
        print("Attempting to load model in 8-bit (bitsandbytes) if available...")
        bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)  # requires bitsandbytes
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            quantization_config=bnb_cfg
        )
        gen_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto", max_new_tokens=512, do_sample=True, temperature=0.2, return_full_text=False)
        llm = HuggingFacePipeline(pipeline=gen_pipeline)
        print("Loaded model with 8-bit quantization.")
        return
    except Exception as e:
        print("8-bit load failed:", e)
        try:
            print("Falling back to float16 load on GPU (if available)...")
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype="auto")
            gen_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto", max_new_tokens=512, do_sample=True, temperature=0.2, return_full_text=False)
            llm = HuggingFacePipeline(pipeline=gen_pipeline)
            print("Loaded model with auto dtype (likely float16 on GPU).")
            return
        except Exception as e2:
            print("float16 load failed:", e2)
            print("As a last resort, you may need to use a smaller model or run via the HF Inference API.")
            raise e2

# Don't auto-load here to let user run explicitly if they want.
print('Model loader ready. Run `load_model(model_name)` to load the generator (may take time and GPU RAM).')


Model loader ready. Run `load_model(model_name)` to load the generator (may take time and GPU RAM).


In [5]:
# Build retriever from current documents stored in memory/FAISS.
def build_retriever():
    global retriever
    if not documents:
        print("No documents present yet — retriever will be None.")
        return None
    # Create langchain documents
    doc_objs = [Document(page_content=t) for t in documents]
    hf_emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    db = LCFAISS.from_documents(doc_objs, hf_emb)
    retriever = db.as_retriever(search_kwargs={"k": 3})
    print(f"Retriever built over {len(documents)} chunks.")
    return retriever

# Build qa chain (conversational) using memory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")
def build_qa_chain():
    global qa_chain
    if retriever is None:
        print("Retriever not ready — cannot build QA chain.")
        return None
    if llm is None:
        print("LLM not loaded — cannot build QA chain.")
        return None
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory,
        return_source_documents=True,
        output_key="answer"  # Explicitly specify which output to use
    )
    print("QA chain built (conversational).")
    return qa_chain

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19176\3958637043.py:16: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")


In [6]:
# Gradio chat function that returns messages-format history (list of {"role","content"} dicts)
def chat_fn(user_message, history):
    try:
        # make sure history is a list
        history = history or []
        if qa_chain is None:
            # if QA chain not built, ask user to upload and load model
            history.append({"role":"assistant","content":" Please load the model first to enable chatting."})
            return history, ""  # clear input box
        # Query the conversational chain
        result = qa_chain({"question": user_message})
        answer = result.get("answer") or result.get("result") or result.get("output_text") or str(result)
        # Append user + assistant messages
        history.append({"role":"user","content":user_message})
        history.append({"role":"assistant","content":answer})
        return history, ""  # clear input box
    except Exception as e:
        import traceback; traceback.print_exc()
        history.append({"role":"assistant","content":f"Error: {e}"})
        return history, ""

In [7]:
# Build the Gradio interface. The upload button calls process_pdf which rebuilds retriever+qa_chain.
with gr.Blocks() as demo:
    gr.Markdown("## RAG PDF Chatbot (FAISS + MiniLM + Mistral)")
    with gr.Row():
        upload = gr.File(label="Upload PDF", file_types=[".pdf"])
        upload_status = gr.Textbox(label="Upload status")
    with gr.Row():
        model_btn = gr.Button("Load Mistral model (may take time & VRAM)")
        model_status = gr.Textbox(label="Model status")
    chatbot = gr.Chatbot(label="Chat", type="messages")
    txt = gr.Textbox(placeholder="Ask questions about uploaded PDFs...")
    clear = gr.Button("Clear chat")
    
    # Wiring
    upload.upload(process_pdf, inputs=upload, outputs=upload_status)
    def _load_model(btn):
        try:
            model_status.value = "Loading..."
        except:
            pass
        try:
            load_model(model_name)
            model_status.value = "Model loaded."
            # after model load, if documents already exist, build retriever+qa_chain
            global retriever, qa_chain
            retriever = build_retriever()
            qa_chain = build_qa_chain()
        except Exception as e:
            model_status.value = f'Error loading model: {e}'
    model_btn.click(_load_model, inputs=[model_btn], outputs=[model_status])
    
    txt.submit(chat_fn, [txt, chatbot], [chatbot, txt])
    clear.click(lambda: [], None, chatbot, queue=False)

demo.launch(share=False, server_port=None)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
